# 第19章 时间序列基础——平稳性与自相关

> **动机先行**: 前面18章的所有数学工具——概率、统计、线性代数、优化——都在处理"某一时刻"的数据。50只股票的日收益率被当作50个独立的数字，昨天的收益率和今天的收益率在数学上没有区别。但金融数据是有"记忆"的: 大波动之后往往跟着大波动（波动率聚集），趋势一旦形成会持续一段时间（动量效应），今天的市场情绪会部分延续到明天。**时间序列分析正是为了建模这种"今天与昨天的关系"而生的数学分支。** 本章建立整个第五阶段的基础概念——平稳性、自相关、白噪声——它们之于时间序列，就如同期望和方差之于概率统计: 是最底层的描述语言。
>
> **量化实战定位**: 平稳性检验是每个量化策略的第一道关卡——非平稳的价格序列不能直接用于回归（会导致伪回归），ACF/PACF 图是识别 ARIMA 模型阶数的标准诊断工具，Ljung-Box 检验则用于验证"我们是否已经提取了数据中的所有可预测结构"。这些概念构成了第20-22章 ARIMA/GARCH/协整的数学地基。

---

## 19.1 动机: 从"截面"到"时间"——数据的顺序开始重要了

回顾之前所有章节的数据使用方式: 搜集 $T$ 天的日收益率 $\{r_1, r_2, \dots, r_T\}$，计算均值 $\bar{r} = \frac{1}{T}\sum r_t$，计算标准差，跑回归。在这个过程中，**数据的顺序无关紧要**——打乱 $r_t$ 的排列顺序，均值不变、方差不变、协方差不变。

但金融直觉告诉你这不对。如果昨天涨了2%，今天更可能涨还是跌？连涨三天后，第四天的预期收益率是否不同？这些问题的答案取决于**数据的顺序**——时间序列分析正是将这种顺序信息纳入数学框架的方法。

一个时间序列 $\{y_t\}_{t=1}^{T}$ 的数学描述需要回答三个层次的问题:

1. **它稳定吗？** 序列的统计特性（均值、方差）是否随时间变化？（→ 平稳性）
2. **它有记忆吗？** 今天的 $y_t$ 与昨天的 $y_{t-1}$ 有多大关联？与一周前的 $y_{t-5}$ 呢？（→ 自相关）
3. **记忆有多长？** 自相关在多长的滞后期后衰减到零？（→ ACF/PACF 诊断）

这三个问题是后续 ARIMA 建模、GARCH 波动率建模、协整检验的通用起点。

---

## 19.2 平稳性: 时间序列分析的"身份证"

### 19.2.1 什么是平稳性？

**平稳性 (Stationarity)** 意味着: 序列的统计特性**不随时间推移而改变**。换个角度看任何一段子序列，它的均值、方差、自相关结构应该大致相同。

形式化定义——**弱平稳 (Covariance Stationary)** 要求三个条件:

1. **均值恒定**: $E[y_t] = \mu$（不随 $t$ 变化）
2. **方差恒定**: $Var(y_t) = \sigma^2$（不随 $t$ 变化）
3. **自协方差只依赖于滞后距离**: $Cov(y_t, y_{t-k})$ 只依赖于 $k$（间隔多少期），不依赖于 $t$（从哪一期开始）

![平稳与非平稳序列对比: 左图均值回复型平稳序列始终围绕0波动,方差稳定; 右图随机游走持续漂移,方差随时间发散。](images/ch19_fig1_stationarity.png)

**金融实例**: 
- 日对数收益率 $\{r_t\}$ 通常是**平稳**的——均值约等于零，波动虽有聚类但长期方差稳定。
- 股票价格 $\{P_t\}$ 是**非平稳**的——今天的价格是 $P_t = P_0 \times e^{r_1+\cdots+r_t}$，方差随时间累加而发散。
- 带趋势的序列（如GDP）是非平稳的，但**差分**一次后通常变平稳。

### 19.2.2 为什么平稳性如此关键？——三个层面的数学后果

如果序列非平稳，整个统计推断大厦的地基会从三个层面崩塌:

**层面一——伪回归 (Spurious Regression)**: 两个独立的随机游走做回归，也会得到"显著"的 $R^2$ 和 t 统计量。Granger & Newbold (1974) 的模拟实验表明，两个无关的随机游走在 5% 显著性水平下，"显著相关"的概率高达 75%！数学原因: $I(1)$ 序列的样本矩(均值、方差、协方差)不收敛于总体参数——大数定律和中心极限定理的前提被破坏，t 统计量不再服从 t 分布。事实上, 当 $T \to \infty$, 伪回归的 $R^2$ 收敛于一个非退化的随机变量(而非零), t 统计量发散到无穷——OLS 会"自信地"报告一个虚假的显著关系。

**层面二——预测方差发散**: 对于随机游走 $y_t = y_{t-1} + \varepsilon_t$, $h$ 步预测的方差为 $Var(y_{T+h} \mid y_T) = h\sigma^2$——随预测期 $h$ 线性发散。而平稳 AR(1) 的预测方差为 $(1-\phi^{2h})\sigma^2/(1-\phi^2)$——有上界。这意味着非平稳序列的长期预测区间无限宽——毫无实用价值。

**层面三——大数定律失效**: 样本均值 $\bar{y} = \frac{1}{T}\sum y_t$ 不收敛到常数。对于 $I(1)$ 序列, $\bar{y} = O_p(T^{1/2})$——样本均值本身的方差随 $T$ 增大, 而非减小。

> **量化实践准则**: 在做任何回归、相关分析、因子检验之前，先用 ADF 检验确认序列是平稳的。如果价格是 $I(1)$, 取对数差分（$I(0)$ 的收益率）；如果收益率仍不平稳，考虑结构突变或使用协整框架(第22章)。

### 19.2.3 ADF 检验的数学原理

**单位根 (Unit Root)** 的概念来自 AR(1) 的表示: $y_t = \phi y_{t-1} + \varepsilon_t$。特征方程 $\phi - z = 0$ 的根 $z = \phi$。当 $|\phi| < 1$ 时根在单位圆外(平稳); 当 $|\phi| = 1$ 时根恰好在单位圆上(**单位根**)。这就是"单位根检验"名称的来源。

**ADF 检验 (Augmented Dickey-Fuller Test)** 将 AR(1) 方程变形为:

$$\Delta y_t = \alpha + \beta t + \gamma y_{t-1} + \sum_{i=1}^{p} \delta_i \Delta y_{t-i} + \varepsilon_t$$

其中 $\gamma = \phi - 1$。$H_0: \gamma = 0$(即 $\phi = 1$, 有单位根, 非平稳); $H_1: \gamma < 0$($\phi < 1$, 平稳)。

**关键时刻——为什么不能用 t 分布？** 在 $H_0$ 下, $\hat{\gamma}$ 的渐近分布不是标准正态, 而是**Dickey-Fuller 分布**——一个有偏的、左尾比 t 分布更靠左的非标准分布。这就是为什么 ADF 检验的临界值(约 -2.86 在 5% 水平)比标准正态分位数(-1.65)严格得多。软件(如 `statsmodels`)会自动使用正确的临界值, 但理解这个区别能帮你避免"用t检验判断平稳性"的常见错误。

**滞后阶数 $p$ 的选择**: 加入 $\Delta y_t$ 的滞后项是为了"漂白"残差——确保 $\varepsilon_t$ 是白噪声。$p$ 通常通过 AIC 或 BIC 自动选择(`autolag='AIC'`)。

### 19.2.4 量化实战: ADF 检验——判断序列是否平稳
1. 估计回归: $\Delta y_t = \alpha + \beta t + \gamma y_{t-1} + \sum_{i=1}^{p} \delta_i \Delta y_{t-i} + \varepsilon_t$
2. 检验 $\gamma = 0$（原假设: 存在单位根）
3. 若 p 值 < 0.05，拒绝原假设 → 序列平稳

用真实A股数据对比**价格（非平稳）**和**收益率（平稳）**:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

csv_path = 'data/index_data_7_20210601_20260531.csv'
df = pd.read_csv(csv_path, parse_dates=['time'])
hs300 = df[df['thscode'] == '000300.SH'].set_index('time')['close']
log_price = np.log(hs300)
log_ret = log_price.diff().dropna()

# ADF 检验
def adf_test(series, name):
    result = adfuller(series.dropna(), autolag='AIC')
    print(f"{name}:")
    print(f"  ADF 统计量 = {result[0]:.4f}")
    print(f"  p 值 = {result[1]:.4f}")
    print(f"  {'→ 序列平稳 (拒绝单位根)' if result[1] < 0.05 else '→ 序列非平稳 (不能拒绝单位根)'}")
    return result

fig, axes = plt.subplots(2, 1, figsize=(13, 7))
axes[0].plot(log_price.index, log_price, color='#2980B9', linewidth=1)
axes[0].set_title('沪深300对数价格 (非平稳)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('log(价格)'); axes[0].grid(True, alpha=0.3)
axes[1].plot(log_ret.index, log_ret, color='#E74C3C', linewidth=0.8)
axes[1].axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
axes[1].set_title('沪深300对数收益率 (平稳)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('log 收益率'); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n=== ADF 检验结果 ===")
adf_test(log_price, "对数价格")
adf_test(log_ret, "对数收益率")

**运行结果**:
```
=== ADF 检验结果 ===
对数价格:
  ADF 统计量 = -1.9883
  p 值 = 0.2917
  → 序列非平稳 (不能拒绝单位根)
对数收益率:
  ADF 统计量 = -34.2210
  p 值 = 0.0000
  → 序列平稳 (拒绝单位根)
```

> **关键收获**: 价格 p 值 0.52（远大于 0.05），不能拒绝"存在单位根"——价格是典型的非平稳序列。收益率的 p 值 $\approx 0$，强烈拒绝单位根——可以安全地用于回归和建模。**处理非平稳序列的第一反应应该是"差分"。**

---

## 19.3 自相关: 过去如何影响现在

### 19.3.1 自协方差与自相关系数

第9章定义了两个不同变量之间的协方差 $Cov(X, Y)$。**自协方差 (Autocovariance)** 是同一个序列在**不同时间**的协方差:

$$\boxed{\gamma_k = Cov(y_t, y_{t-k}) = E[(y_t - \mu)(y_{t-k} - \mu)]}$$

其中 $k$ 是**滞后期数 (Lag)**。$\gamma_0 = Var(y_t)$ 是自身的方差。

**自相关系数 (ACF, Autocorrelation Function)** 将自协方差标准化到 $[-1, 1]$:

$$\boxed{\rho_k = \frac{\gamma_k}{\gamma_0} = \frac{Cov(y_t, y_{t-k})}{Var(y_t)}}$$

- $\rho_1$: 今天与昨天的相关性——如果 $\rho_1 > 0$，意味着"昨天涨，今天倾向于涨"（正自相关，动量效应）；如果 $\rho_1 < 0$，意味着"昨天涨，今天倾向于跌"（负自相关，均值回复）。
- $\rho_0 = 1$ 永远是 1（自己与自己的相关系数是 1）。
- 平稳序列的 $\rho_k$ 随 $k$ 增加而衰减。

**样本自相关系数**（用有限数据估计）:

$$\hat{\rho}_k = \frac{\sum_{t=k+1}^{T} (y_t - \bar{y})(y_{t-k} - \bar{y})}{\sum_{t=1}^{T} (y_t - \bar{y})^2}$$

在 $H_0: \rho_k = 0$ 下, $\hat{\rho}_k \stackrel{approx}{\sim} N(0, 1/T)$——这就是 ACF 图中 95% 置信带 $\pm 1.96/\sqrt{T}$ 的理论依据。

**Yule-Walker 方程: 从 ACF 到 AR 系数的桥梁**。对于 AR(p) 过程, 自相关系数满足:

$$\rho_k = \phi_1 \rho_{k-1} + \phi_2 \rho_{k-2} + \cdots + \phi_p \rho_{k-p}, \quad k = 1, 2, \dots$$

这 $p$ 个方程组成的线性系统叫 **Yule-Walker 方程**。写成矩阵形式:

$$\begin{pmatrix} \rho_0 & \rho_1 & \cdots & \rho_{p-1} \\ \rho_1 & \rho_0 & \cdots & \rho_{p-2} \\ \vdots & \vdots & \ddots & \vdots \\ \rho_{p-1} & \rho_{p-2} & \cdots & \rho_0 \end{pmatrix} \begin{pmatrix} \phi_1 \\ \phi_2 \\ \vdots \\ \phi_p \end{pmatrix} = \begin{pmatrix} \rho_1 \\ \rho_2 \\ \vdots \\ \rho_p \end{pmatrix}$$

用样本 ACF $\hat{\rho}_k$ 替代理论 $\rho_k$, 解此线性系统就得到 AR 系数的**矩估计 (Method of Moments)**。这是 AR 模型最早的估计方法——比 MLE 简单, 但不一定最有效。更重要的是, Yule-Walker 方程揭示了 **ACF 和 AR 系数之间的一一对应关系**——给定 $\phi_1, \dots, \phi_p$, 自相关结构完全确定; 反之亦然。

### 19.3.2 ACF 图: 寻找"记忆"的长度

ACF 图是时间序列分析最重要的诊断工具——横轴是滞后期 $k$，纵轴是 $\hat{\rho}_k$。图中通常有两条蓝色虚线表示 95% 置信区间（约 $\pm 1.96/\sqrt{T}$），超出该区间的自相关系数在统计上显著不为零。

三种典型的 ACF 模式:

| ACF 模式 | 含义 | 对应模型 |
|---------|------|---------|
| 截尾 (Cut Off) | 前 $q$ 个显著，之后全部落入置信区间 | MA(q) |
| 拖尾 (Tail Off) | 指数衰减或振荡衰减，缓慢趋近于零 | AR(p) |
| 不衰减 | 一直显著不为零，衰减极慢 | 非平稳（需要差分） |

### 19.3.3 量化实战: 绘制沪深300收益率的 ACF

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

csv_path = 'data/index_data_7_20210601_20260531.csv'
df = pd.read_csv(csv_path, parse_dates=['time'])
hs300_ret = np.log(df[df['thscode']=='000300.SH'].set_index('time')['close']).diff().dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图: 收益率时序
axes[0].plot(hs300_ret.index, hs300_ret, color='#E74C3C', linewidth=0.6)
axes[0].axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
axes[0].set_title('沪深300日对数收益率', fontsize=13, fontweight='bold')
axes[0].set_ylabel('收益率'); axes[0].grid(True, alpha=0.3)

# 右图: ACF
plot_acf(hs300_ret.dropna(), lags=40, ax=axes[1], color='#2980B9')
axes[1].set_title('ACF (自相关函数)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('滞后期 k (天)'); axes[1].set_ylabel('自相关系数')
plt.tight_layout()
plt.show()

# 关键滞后期
T = len(hs300_ret)
from statsmodels.tsa.stattools import acf
acf_vals, ci = acf(hs300_ret.dropna(), nlags=5, alpha=0.05)
print("前5阶自相关系数:")
for k in range(1, 6):
    sig = '显著' if abs(acf_vals[k]) > ci[k,1] - acf_vals[k] else '不显著'
    print(f"  rho_{k} = {acf_vals[k]:+.4f}  ({sig})")
print(f"\n95% 置信区间约: +/- {1.96/np.sqrt(T):.4f}")

**运行结果**:
```
前5阶自相关系数:
  rho_1 = +0.0149  (不显著)
  rho_2 = +0.0068  (不显著)
  rho_3 = -0.0266  (不显著)
  rho_4 = +0.0365  (不显著)
  rho_5 = -0.0129  (不显著)

95% 置信区间约: +/- 0.0564
```

> **关键收获**: 沪深300的日收益率几乎没有显著的自相关——今天的涨跌与昨天基本无关。这是**有效市场假说**的统计证据: 价格变动是近似随机的，无法用"昨天涨了"来预测"今天涨不涨"。但这不意味着收益率序列是"白噪声"——下一节将看到，**收益率的平方**具有很强的自相关（波动率聚集），这是第21章 GARCH 模型的基础。

---

## 19.4 偏自相关: 剥离间接影响

### 19.4.1 PACF 的直觉

ACF 测量的是 $y_t$ 与 $y_{t-k}$ 之间的**总相关**——包含所有中间路径的间接影响。例如 $\rho_2$ 可能是:
- $y_t$ 与 $y_{t-2}$ 的**直接**相关
- $y_t \leftarrow y_{t-1} \leftarrow y_{t-2}$ 的**间接**传导（今天的变动→昨天的变动→前天的变动）

**偏自相关系数 (PACF, Partial Autocorrelation Function)** 剥离了中间变量的影响，只保留**直接效应**:

$$\boxed{\phi_{kk} = Corr(y_t, y_{t-k} \mid y_{t-1}, \dots, y_{t-k+1})}$$

是"在已知中间所有值 $y_{t-1}$ 到 $y_{t-k+1}$ 的条件下，$y_t$ 与 $y_{t-k}$ 的净相关"。

### 19.4.2 量化实战: ACF 与 PACF 对比

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

csv_path = 'data/index_data_7_20210601_20260531.csv'
df = pd.read_csv(csv_path, parse_dates=['time'])
hs300_ret = np.log(df[df['thscode']=='000300.SH'].set_index('time')['close']).diff().dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_acf(hs300_ret.dropna(), lags=30, ax=axes[0], color='#2980B9')
axes[0].set_title('ACF (自相关) —— 含间接效应', fontsize=13, fontweight='bold')
plot_pacf(hs300_ret.dropna(), lags=30, ax=axes[1], color='#E74C3C', method='ywm')
axes[1].set_title('PACF (偏自相关) —— 仅直接效应', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# 对比数值
from statsmodels.tsa.stattools import acf, pacf
a = acf(hs300_ret.dropna(), nlags=5)
p = pacf(hs300_ret.dropna(), nlags=5, method='ywm')
print("滞后期   ACF       PACF")
print("-" * 28)
for k in range(1, 6):
    print(f"  k={k}    {a[k]:+.4f}    {p[k]:+.4f}")

**运行结果**:
```
滞后期   ACF       PACF
----------------------------
  k=1    +0.0149    +0.0149
  k=2    +0.0068    +0.0065
  k=3    -0.0266    -0.0268
  k=4    +0.0365    +0.0373
  k=5    -0.0129    -0.0137
```

> **注意**: 对于收益率本身，ACF 和 PACF 差异很小——因为各阶自相关都不显著，间接传导的路径几乎不存在。但在波动率建模（第21章 GARCH）中，收益率平方的 ACF 和 PACF 会表现出显著且持续的自相关，两者差异也更明显。

---

## 19.5 白噪声: 最简单的时间序列基准

### 19.5.1 什么是白噪声？

**白噪声 (White Noise)** 是时间序列的"零假设"——序列没有任何可预测的结构:

$$\boxed{y_t \sim WN(0, \sigma^2): \quad E[y_t] = 0, \quad Var(y_t) = \sigma^2, \quad Cov(y_t, y_{t-k}) = 0 \; (\forall k \neq 0)}$$

三个条件: 均值为零、方差恒定、任意两期之间**完全不相关**。它的 ACF 在 $k \geq 1$ 时理论上全为零。

在量化中，白噪声是重要的基准模型:
- 如果你建立了一个预测模型（如 ARIMA），残差应该是白噪声——如果残差还有自相关结构，说明模型没有提取完所有信息，需要改进。
- "有效市场"的一个统计表述就是: **收益率序列接近于白噪声**（但不完全等同于白噪声，因为波动率仍有结构）。

### 19.5.2 Ljung-Box 检验: 联合检验的数学推导

单独检验 $\rho_1$、$\rho_2$ 的显著性会遇到多重检验问题——检验 20 个滞后, 即使序列真的是白噪声, 也期望有 1 个"碰巧显著"(20 × 5% = 1)。**Ljung-Box 检验 (Ljung & Box, 1978)** 用一个统计量联合检验前 $m$ 个自相关系数是否**同时**为零:

$$H_0: \rho_1 = \rho_2 = \cdots = \rho_m = 0 \quad \text{(序列是白噪声)}$$

**推导思路**: 在 $H_0$ 下, $\sqrt{T}(\hat{\rho}_1, \dots, \hat{\rho}_m)^T \stackrel{d}{\to} N(\mathbf{0}, \mathbf{I}_m)$——样本自相关是渐近独立的标准正态。原 Box-Pierce (1970) 统计量为 $Q_{BP} = T\sum_{k=1}^{m}\hat{\rho}_k^2 \sim \chi^2_m$。Ljung & Box (1978) 发现小样本下 $Q_{BP}$ 的卡方近似不准确, 提出了修正版:

$$\boxed{Q(m) = T(T+2) \sum_{k=1}^{m} \frac{\hat{\rho}_k^2}{T-k} \stackrel{H_0}{\sim} \chi^2_m}$$

修正因子 $(T+2)/(T-k)$ 使小样本下的分布更接近卡方——即使 $T$ 只有几十个观测, Ljung-Box 检验也比 Box-Pierce 可靠。

**应用逻辑**: 
- p 值 $> 0.05$ → 不能拒绝 $H_0$(白噪声) → 数据中**没有显著的可预测结构** → ARMA 建模可能没有意义
- p 值 $< 0.05$ → 拒绝白噪声 → 存在自相关 → 可以尝试 ARMA 建模(第20章)
- **陷阱**: p 值 > 0.05 只是说"不能拒绝白噪声", 不等于"序列就是白噪声"——样本量小时检验功效低, 可能漏掉弱自相关

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.diagnostic import acorr_ljungbox

csv_path = 'data/index_data_7_20210601_20260531.csv'
df = pd.read_csv(csv_path, parse_dates=['time'])
hs300_ret = np.log(df[df['thscode']=='000300.SH'].set_index('time')['close']).diff().dropna()

# Ljung-Box 检验 (不同 m)
print("沪深300日收益率 Ljung-Box 检验:")
for m in [5, 10, 20]:
    result = acorr_ljungbox(hs300_ret.dropna(), lags=[m], return_df=True)
    pval = result['lb_pvalue'].values[0]
    print(f"  m={m:2d}: Q={result['lb_stat'].values[0]:.2f}, p={pval:.4f}  "
          f"{'→ 白噪声' if pval > 0.05 else '→ 存在自相关'}")

# 对比: 收益率的绝对值 (波动率代理)
abs_ret = np.abs(hs300_ret)
print("\n|收益率| (波动率代理) Ljung-Box 检验:")
for m in [5, 10, 20]:
    result = acorr_ljungbox(abs_ret.dropna(), lags=[m], return_df=True)
    pval = result['lb_pvalue'].values[0]
    print(f"  m={m:2d}: Q={result['lb_stat'].values[0]:.2f}, p={pval:.4f}  "
          f"{'→ 白噪声' if pval > 0.05 else '→ 存在自相关 (波动率聚集)'}")

**运行结果**:
```
沪深300日收益率 Ljung-Box 检验:
  m= 5: Q=3.01, p=0.6989  → 白噪声
  m=10: Q=11.35, p=0.3310  → 白噪声
  m=20: Q=26.40, p=0.1530  → 白噪声

|收益率| (波动率代理) Ljung-Box 检验:
  m= 5: Q=191.46, p=0.0000  → 存在自相关 (波动率聚集)
  m=10: Q=224.09, p=0.0000  → 存在自相关 (波动率聚集)
  m=20: Q=242.05, p=0.0000  → 存在自相关 (波动率聚集)
```

> **量化解读**: 收益率本身接近白噪声（p > 0.6，不能拒绝"无自相关"），意味着无法用过去的收益率线性预测未来——这正是有效市场的体现。但波动率（以 $|r_t|$ 代理）具有强烈且持续的自相关（p ≈ 0），验证了"大波动后跟随大波动"的波动率聚集现象。**Alpha 不在于预测涨跌方向，而在于预测波动率和相关性结构。**

---

## 19.6 核心公式速查

> 本节是前述各节公式的集中汇总, 供复习和查阅使用.

| 概念 | 公式 | 量化意义 |
|------|------|---------|
| 弱平稳 | $E[y_t]=\mu$, $Var(y_t)=\sigma^2$, $Cov(y_t,y_{t-k})$ 仅依赖 $k$ | 统计特性不随时间改变——建模的前提条件 |
| 自协方差 | $\gamma_k = Cov(y_t, y_{t-k})$ | 度量序列"k天前"与"今天"的协变关系 |
| 自相关系数 | $\rho_k = \gamma_k / \gamma_0$ | 标准化到[-1,1]的自相关度量 |
| 样本 ACF | $\hat{\rho}_k = \frac{\sum(y_t-\bar{y})(y_{t-k}-\bar{y})}{\sum(y_t-\bar{y})^2}$ | 从有限数据估计自相关 |
| PACF | $\phi_{kk} = Corr(y_t, y_{t-k} \mid y_{t-1},\dots,y_{t-k+1})$ | 剥离中间变量的间接影响 |
| 白噪声 | $y_t \sim WN(0,\sigma^2)$: $E[y_t]=0, Var(y_t)=\sigma^2, \rho_k=0$ | 无预测结构的基准模型 |
| Ljung-Box | $Q(m) = T(T+2)\sum_{k=1}^{m}\frac{\hat{\rho}_k^2}{T-k}$ | 联合检验前m个自相关是否为零 |
| ADF 检验 | 检验 $\gamma=0$ 在 $\Delta y_t = \alpha + \gamma y_{t-1} + \cdots$ | 判断序列是否有单位根(非平稳) |

---

## 19.7 本章小结

| 概念 | 核心要点 | 量化意义 |
|------|---------|---------|
| 平稳性 | 均值、方差、自协方差不随时间改变 | 回归/预测/大数定律的前提——价格不平稳, 收益率平稳 |
| ADF 检验 | H0: 有单位根(非平稳) | 每个量化分析的第一步: 确认序列平稳 |
| ACF | 滞后期 k 的总自相关 | 识别 MA 模型阶数; 判断序列是否有"记忆" |
| PACF | 剥离中间变量后的净自相关 | 识别 AR 模型阶数; 区分直接vs间接影响 |
| 白噪声 | 均值为0, 方差恒定, 无自相关 | 模型残差应接近白噪声——否则还有信息没提取完 |
| Ljung-Box | 联合检验 H0: ρ1=...=ρm=0 | 正式检验"序列是否可预测"; 收益率→白噪声, 波动率→非白噪声 |

**从本章走向下一章**:
- 第20章将用 ACF 和 PACF 的截尾/拖尾模式来识别 ARMA 模型的阶数——AR(p) 的 PACF 截尾, MA(q) 的 ACF 截尾。你将学到如何用 ARIMA 模型预测收益率（并理解为什么这个预测通常效果有限）。

---

## 19.8 练习题

### 数学推导

**题1——平稳性的验证**:

(a) 证明白噪声序列 $y_t = \varepsilon_t$（$\varepsilon_t$ i.i.d. $N(0,\sigma^2)$）满足弱平稳的三个条件。

(b) 随机游走 $y_t = y_{t-1} + \varepsilon_t$（$y_0=0$）的方差 $Var(y_t)$ 是多少？它随 $t$ 增大是收敛还是发散？由此判断随机游走是否平稳。

(c) AR(1) 过程 $y_t = \phi y_{t-1} + \varepsilon_t$。证明当 $|\phi| < 1$ 时序列平稳，当 $|\phi| = 1$ 时退化（随机游走）为非平稳。

**题2——ACF 的性质**:

(a) 对于白噪声序列，证明理论 ACF 在 $k \geq 1$ 时 $\rho_k = 0$。

(b) 样本 ACF 在 $H_0: \rho_k = 0$ 下近似服从 $N(0, 1/T)$。推导 ACF 图中 95% 置信区间的公式 $\pm 1.96/\sqrt{T}$。

(c) 为什么小样本下 $\hat{\rho}_k$ 可能偏离零很多，即使真实 $\rho_k = 0$？这对解读短时间序列的 ACF 图有什么影响？

**题3——伪回归的数学直觉**:

(a) 两个独立随机游走 $X_t = X_{t-1} + u_t$, $Y_t = Y_{t-1} + v_t$。回归 $Y_t = \beta_0 + \beta_1 X_t + \varepsilon_t$。为什么 $\beta_1$ 即使在 $X$ 和 $Y$ 完全无关时也经常"显著"？(提示: 考虑两个序列的共同趋势会"碰巧"同向。)

(b) 如果对差分后的序列 $\Delta Y_t$ 回归到 $\Delta X_t$，伪回归问题是否还存在？为什么？

### 编程实践

**题4——多资产平稳性检验**: 对 `stock_data_50` 中 10 只不同行业的股票，分别对**(i) 对数价格**和**(ii) 对数收益率**做 ADF 检验。汇总结果: 多少只股票的价格非平稳（p>0.05）？多少只股票的收益率平稳（p<0.05）？这个结果支持"价格非平稳、收益率平稳"的一般规律吗？

**题5——ACF 模式识别**: 

(a) 模拟一个 AR(1) 过程 $y_t = 0.7 y_{t-1} + \varepsilon_t$ 和一个 MA(1) 过程 $y_t = \varepsilon_t + 0.7 \varepsilon_{t-1}$（各500期）。画出两者的 ACF 和 PACF 图（共4张子图），观察: AR(1) 的 ACF 拖尾、PACF 截尾；MA(1) 的 ACF 截尾、PACF 拖尾。

(b) 用 Ljung-Box 检验判断 AR(1) 的残差是否是白噪声——这与 (a) 中观察到的 ACF 模式一致吗？

---

## 19.9 参考文献

1. **Box, G. E. P., Jenkins, G. M., Reinsel, G. C., & Ljung, G. M.** (2015). *Time Series Analysis: Forecasting and Control* (5th ed.). Wiley. （时间序列分析的"圣经"——本章的 ACF、PACF、平稳性概念均源自 Box-Jenkins 方法论）

2. **Tsay, R. S.** (2010). *Analysis of Financial Time Series* (3rd ed.). Wiley. （金融时间序列的标准教材——第2章对平稳性和自相关的金融数据特性有系统讨论）

3. **Granger, C. W. J., & Newbold, P.** (1974). Spurious regressions in econometrics. *Journal of Econometrics*, 2(2), 111-120. （伪回归问题的经典论文——两个随机游走回归的模拟实验肇始于此）

4. **Dickey, D. A., & Fuller, W. A.** (1979). Distribution of the estimators for autoregressive time series with a unit root. *Journal of the American Statistical Association*, 74(366), 427-431. （ADF 检验的原始论文——单位根检验的标准方法）

5. **Ljung, G. M., & Box, G. E. P.** (1978). On a measure of lack of fit in time series models. *Biometrika*, 65(2), 297-303. （Ljung-Box 检验的原始论文——Q 统计量的推导与分布）

---

> **愿我们都能在数字与代码之间, 找到理解市场的那把钥匙.**
>
> *数学的理解没有捷径, 量化的能力无法外包.*